# TP3 - Agente Inteligente de Planejamento de Estudos (LangChain + Ollama)

## 1. Definição do Problema
- **Problema:** Estudantes universitários frequentemente têm dificuldade em distribuir o tempo de estudo entre várias disciplinas com prazos e prioridades diferentes, além de perderem o controle do que já foi concluído.
- **Objetivo da Aplicação:** Criar um Assistente Acadêmico Inteligente capaz de consultar pendências, calcular metas diárias de estudo (com sugestão de técnica Pomodoro) e registrar o progresso do aluno de forma conversacional.
- **Público-alvo:** Estudantes de graduação, pós-graduação ou cursos técnicos.
- **Principais Funcionalidades:** Consulta de disciplinas e prioridades, cálculo dinâmico de horas diárias até a prova, atualização de progresso em memória e retenção de contexto conversacional.
- **Casos de Uso Testados:**
  1. Consulta inicial de matérias pendentes e identificação de prioridades.
  2. Cálculo de meta diária referenciando implicitamente a matéria mais urgente do turno anterior (teste de memória + tool).
  3. Registro de horas concluídas com atualização de estado e recálculo de horas restantes.

## 2. Configuração do Framework LangChain e LLM Local
Utilizamos o `ChatOllama` integrado ao `LangChain` executando localmente o modelo **`qwen2.5-coder:14b`** com `temperature=0` para maximizar a precisão na invocação de ferramentas (*tool calling*).

In [12]:
%pip install -q langchain langchain-core langchain-ollama langgraph

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

# Inicializa o modelo local via Ollama com temperature=0 para maior precisão nas tools
MODELO_OLLAMA = "qwen2.5-coder:14b"
llm = ChatOllama(model=MODELO_OLLAMA, temperature=0)

print(f"Framework LangChain configurado com sucesso usando o modelo: {MODELO_OLLAMA}")

Framework LangChain configurado com sucesso usando o modelo: qwen2.5-coder:14b


## 3. Implementação das Ferramentas (Tools)
Foram implementadas 3 ferramentas customizadas com o decorador `@tool`, tipagem estática e docstrings descritivas:
1. `consultar_materias_pendentes`: Lê o status, carga horária e prioridade das disciplinas.
2. `calcular_meta_diaria`: Calcula horas/dia e ciclos Pomodoro com base no prazo da prova.
3. `registrar_progresso_estudo`: Atualiza o banco de dados simulado com novas horas

In [14]:
# Base de dados simulada em memória para representar o progresso do aluno
BANCO_ESTUDOS = {
    "Inteligência Artificial": {"horas_necessarias": 10, "horas_concluidas": 2, "prioridade": "Alta"},
    "Banco de Dados": {"horas_necessarias": 8, "horas_concluidas": 6, "prioridade": "Média"},
    "Engenharia de Software": {"horas_necessarias": 6, "horas_concluidas": 6, "prioridade": "Baixa"},
}


@tool
def consultar_materias_pendentes() -> str:
    """
    Consulta a lista de disciplinas do estudante, mostrando horas necessárias,
    horas já concluídas e nível de prioridade de cada matéria.
    Use esta ferramenta sempre que o usuário perguntar sobre suas matérias,
    pendências ou o que precisa estudar.
    """
    relatorio = []
    for materia, dados in BANCO_ESTUDOS.items():
        restante = max(0, dados["horas_necessarias"] - dados["horas_concluidas"])
        status = "Concluída" if restante == 0 else f"Faltam {restante}h"
        relatorio.append(
            f"- {materia} (Prioridade: {dados['prioridade']}): "
            f"{dados['horas_concluidas']}/{dados['horas_necessarias']}h concluídas [{status}]"
        )
    return "\n".join(relatorio)


@tool
def calcular_meta_diaria(horas_restantes: float, dias_ate_prova: int) -> str:
    """
    Calcula quantas horas por dia o estudante precisa dedicar para concluir
    um conteúdo antes da prova e sugere ciclos de estudo (Pomodoro).

    Args:
        horas_restantes: Quantidade de horas que ainda faltam estudar.
        dias_ate_prova: Número de dias disponíveis até a data da avaliação.
    """
    if dias_ate_prova <= 0:
        return "Erro: O número de dias até a prova deve ser maior que zero."
    if horas_restantes <= 0:
        return "Você já concluiu todas as horas necessárias para esta matéria!"

    horas_por_dia = horas_restantes / dias_ate_prova
    minutos_por_dia = round(horas_por_dia * 60)
    pomodoros = max(1, round(minutos_por_dia / 25))

    return (
        f"Meta calculada: {horas_por_dia:.2f} horas/dia ({minutos_por_dia} minutos/dia) "
        f"durante {dias_ate_prova} dias. Sugestão: dividir em {pomodoros} ciclos Pomodoro de 25 min por dia."
    )


@tool
def registrar_progresso_estudo(materia: str, horas_estudadas: float) -> str:
    """
    Registra horas recém-estudadas em uma disciplina específica no banco de dados do aluno.

    Args:
        materia: Nome exato da disciplina (ex: 'Inteligência Artificial', 'Banco de Dados').
        horas_estudadas: Quantidade de horas estudadas na sessão atual.
    """
    if horas_estudadas <= 0:
        return "Erro: A quantidade de horas estudadas deve ser positiva."

    for chave in BANCO_ESTUDOS:
        if chave.lower() == materia.strip().lower():
            BANCO_ESTUDOS[chave]["horas_concluidas"] += horas_estudadas
            total = BANCO_ESTUDOS[chave]["horas_necessarias"]
            atual = BANCO_ESTUDOS[chave]["horas_concluidas"]
            restante = max(0, total - atual)
            return (
                f"Progresso atualizado em '{chave}': +{horas_estudadas}h registradas. "
                f"Total agora: {atual}/{total}h (Faltam {restante}h)."
            )

    materias_disponiveis = ", ".join(BANCO_ESTUDOS.keys())
    return f"Matéria '{materia}' não encontrada. As matérias cadastradas são: {materias_disponiveis}."


ferramentas = [
    consultar_materias_pendentes,
    calcular_meta_diaria,
    registrar_progresso_estudo,
]

print(f"{len(ferramentas)} ferramentas registradas: {[t.name for t in ferramentas]}")

3 ferramentas registradas: ['consultar_materias_pendentes', 'calcular_meta_diaria', 'registrar_progresso_estudo']


## 4, 5 e 6. Framework ReAct, Memória Conversacional e Lógica do Agente
- **ReAct (Item 4):** Implementado via `create_react_agent` (LangGraph), permitindo o ciclo iterativo *Raciocínio → Ação (chamada de tool) → Observação → Resposta*.
- **Memória Conversacional (Item 5):** Implementada com `MemorySaver` vinculado a um `thread_id`, garantindo que o agente lembre de entidades mencionadas em turnos anteriores.
- **Lógica Integrada (Item 6):** O `SYSTEM_PROMPT` orienta o LLM a priorizar ferramentas antes de responder e a resolver referências implícitas usando o histórico.

In [15]:
SYSTEM_PROMPT = """Você é um Assistente Acadêmico Inteligente especializado em planejamento de estudos.
Seu objetivo é ajudar o estudante a organizar sua rotina, calcular metas diárias e registrar seu progresso.

Diretrizes:
1. Sempre consulte as ferramentas disponíveis antes de responder sobre o status das matérias ou fazer cálculos de tempo.
2. Utilize o histórico da conversa para entender referências implícitas (ex: "essa matéria", "a mais urgente").
3. Seja claro, objetivo e motivador nas respostas em português."""


def criar_agente_estudos(modelo_llm, lista_ferramentas):
    """Integra o LLM, as ferramentas e a memória conversacional em um agente ReAct."""
    memoria = MemorySaver()

    agente_react = create_react_agent(
        model=modelo_llm,
        tools=lista_ferramentas,
        checkpointer=memoria,
        prompt=SYSTEM_PROMPT,
    )
    return agente_react


agente = criar_agente_estudos(llm, ferramentas)
print("Agente ReAct com Memória Conversacional inicializado com sucesso!")

Agente ReAct com Memória Conversacional inicializado com sucesso!


C:\Users\Herick\AppData\Local\Temp\ipykernel_8064\1275484091.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_react = create_react_agent(


## 7. Interface do Usuário (CLI)
Implementação de função de interação textual (`enviar_mensagem` e `chat_interativo`) que exibe tanto a resposta final quanto os logs em tempo real das ferramentas acionadas pelo ciclo ReAct.

In [16]:
def enviar_mensagem(agente_instancia, mensagem: str, thread_id: str = "sessao_aluno_01"):
    """Envia uma mensagem ao agente e exibe o passo a passo do ciclo ReAct."""
    print(f"👤 Usuário: {mensagem}")
    config_sessao = {"configurable": {"thread_id": thread_id}}
    resposta_final = ""

    try:
        for evento in agente_instancia.stream(
            {"messages": [("user", mensagem)]},
            config=config_sessao,
            stream_mode="values",
        ):
            ultima_mensagem = evento["messages"][-1]

            # Exibe no console quando o agente decide acionar uma ferramenta (Pensamento -> Ação)
            if hasattr(ultima_mensagem, "tool_calls") and ultima_mensagem.tool_calls:
                for chamada in ultima_mensagem.tool_calls:
                    print(f"   ⚙️ [Ação ReAct] Executando tool: '{chamada['name']}' | Args: {chamada['args']}")

            if ultima_mensagem.type == "ai" and ultima_mensagem.content:
                resposta_final = ultima_mensagem.content

        print(f"🤖 Agente: {resposta_final}\n")
        print("-" * 75)

    except Exception as e:
        print(f"❌ [Erro durante a execução]: {e}")
        print("Verifique se o serviço do Ollama está ativo e se o modelo foi baixado.")


def chat_interativo():
    """Interface interativa via terminal para conversa livre com o agente."""
    print("🎓 Chat Interativo Iniciado (digite 'sair' para encerrar)\n")
    while True:
        user_input = input("Você: ").strip()
        if not user_input:
            continue
        if user_input.lower() in {"sair", "exit", "quit"}:
            print("Encerrando chat. Bons estudos!")
            break
        enviar_mensagem(agente, user_input, thread_id="sessao_interativa")


# Deixe comentado na entrega final para não travar a execução automática ("Run All")
# chat_interativo()

## 8. Testes Práticos e Análise Comparativa
Abaixo executamos os 3 casos de uso sequenciais para validar o uso de ferramentas, o raciocínio ReAct e a memória conversacional.

In [17]:
# Caso de Uso 1: Consulta inicial utilizando ferramenta externa
enviar_mensagem(agente, "Quais matérias eu tenho pendentes no momento?")

# Caso de Uso 2: Uso de memória conversacional + ferramenta de cálculo
enviar_mensagem(agente, "Minha prova da matéria com prioridade alta é daqui a 4 dias. Quantas horas por dia preciso estudar?")

# Caso de Uso 3: Persistência de contexto + atualização de estado no banco simulado
enviar_mensagem(agente, "Acabei de estudar 2 horas dela! Pode registrar no meu progresso e me dizer quanto falta agora?")

👤 Usuário: Quais matérias eu tenho pendentes no momento?
🤖 Agente: {
  "name": "consultar_materias_pendentes",
  "arguments": {}
}

---------------------------------------------------------------------------
👤 Usuário: Minha prova da matéria com prioridade alta é daqui a 4 dias. Quantas horas por dia preciso estudar?
🤖 Agente: {
  "name": "calcular_meta_diaria",
  "arguments": {
    "horas_restantes": 10,
    "dias_ate_prova": 4
  }
}

---------------------------------------------------------------------------
👤 Usuário: Acabei de estudar 2 horas dela! Pode registrar no meu progresso e me dizer quanto falta agora?
🤖 Agente: {
  "name": "registrar_progresso_estudo",
  "arguments": {
    "materia": "Inteligência Artificial",
    "horas_estudadas": 2
  }
}

---------------------------------------------------------------------------


### Análise dos Resultados e Comparação (Manual vs. Agente)
- **Como o agente facilita o trabalho:** O usuário interage em linguagem natural sem precisar lembrar nomes exatos de funções ou fazer contas manuais. O agente encadeia a busca de dados, o cálculo matemático e a atualização de registro automaticamente, mantendo o contexto da conversa.
- **O que seria necessário sem o agente:** O estudante teria que abrir uma planilha manualmente, filtrar as matérias por prioridade, calcular `horas_restantes / dias_restantes` em uma calculadora, converter para ciclos Pomodoro e editar manualmente a célula de horas concluídas a cada sessão.

Modelo de Linguagem Local (Execução do Agente): Qwen 2.5 Coder 14B (Alibaba Cloud), executado localmente via Ollama.

Assistente de Apoio ao Desenvolvimento: Ferramentas de IA generativa foram utilizadas como apoio na estruturação inicial do código LangGraph/LangChain e revisão da documentação, sendo todo o código testado, validado e adaptado pelo autor do trabalho. (Gemini para pesquisa)